# 📘 Módulo 02: Métodos de Filtrado Avanzado, Búsqueda de Patrones (LIKE / ILIKE) & Subconsultas
### 🔍 Búsqueda Flexible de Texto, Operadores Comodín (`%`, `_`), Escape de Caracteres, Lógica Trivalente y Subconsultas Combinadas
**Asignatura:** Bases de Datos Avanzadas  
**Entorno:** Google Colab / DuckDB (100% In-Memory)  
---
## 🎯 Objetivos de Aprendizaje
1. Dominar el uso de los operadores de comparación de patrones de texto `LIKE` (sensible a mayúsculas) e `ILIKE` (insensible a mayúsculas).
2. Comprender la semántica y uso técnico de los caracteres comodín:
   - `%` (Porcentaje): Coincide con **cero o más caracteres** de cualquier tipo.
   - `_` (Guion bajo): Coincide con **exactamente un único carácter** en una posición fija.
3. Aprender a **escapar caracteres comodín literales** (ej. buscar el carácter `%` o `_` dentro de un texto) usando la cláusula `ESCAPE`.
4. Construir **Subconsultas Complejas combinadas con `LIKE`** para responder preguntas de negocio multinivel.
5. Dominar la **Lógica Trivalente (TRUE, FALSE, UNKNOWN)** ante valores `NULL` al aplicar filtros de texto y funciones como `COALESCE`.

In [1]:
# ==============================================================================
# 🚀 SETUP INICIAL AUTO-CONTENIDO: DUCKDB & GOOGLE COLAB (100% IN-MEMORY)
# ==============================================================================
!pip install duckdb duckdb-engine jupysql -q

import duckdb

%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# Conectar a motor DuckDB en memoria
conn = duckdb.connect(database=':memory:')

# Poblar esquema de datos de prueba
conn.execute("""
CREATE TABLE IF NOT EXISTS clientes_cartera (
    cliente_id INT PRIMARY KEY,
    nombre VARCHAR NOT NULL,
    doc_identidad VARCHAR UNIQUE NOT NULL,
    score_credito INT,
    cupo_aprobado DECIMAL(15,2),
    ciudad VARCHAR,
    email VARCHAR
);
INSERT INTO clientes_cartera VALUES
(1, 'Inversiones Empresariales Bogotá S.A.S.', '900123456-1', 820, 150000000.00, 'Bogotá', 'contacto@inversionesbogota.co'),
(2, 'Construcciones del Valle Ltda.', '890987654-3', 680, 80000000.00, 'Cali', 'gerencia@convalle.com'),
(3, 'María Fernanda Gómez', '1018456789', 750, 25000000.00, 'Medellín', 'mfgomez@email.com'),
(4, 'Comercializadora Caribe S.A.', '800555444-2', 590, 45000000.00, 'Barranquilla', 'ventas@caribesa.co'),
(5, 'Jorge Enrique Restrepo', '79456123', 420, 10000000.00, 'Bogotá', 'jrestrepo@email.com'),
(6, 'Tecnología & Soluciones 50% S.A.S.', '901999888-7', 810, 95000000.00, 'Bogotá', 'info@tech_sol.com.co'),
(7, 'Servicios Logísticos del Eje Ltda.', '830111222-9', 710, 60000000.00, 'Pereira', NULL);

CREATE TABLE IF NOT EXISTS creditos (
    credito_id INT PRIMARY KEY,
    cliente_id INT REFERENCES clientes_cartera(cliente_id),
    monto_desembolsado DECIMAL(15,2),
    saldo_actual DECIMAL(15,2),
    tasa_interes_anual DECIMAL(5,2),
    fecha_desembolso DATE,
    estado VARCHAR
);
INSERT INTO creditos VALUES
(101, 1, 100000000.00, 75000000.00, 16.50, '2023-01-15', 'VIGENTE'),
(102, 1, 40000000.00, 38000000.00, 18.00, '2023-06-10', 'VIGENTE'),
(103, 2, 60000000.00, 60000000.00, 19.50, '2023-09-01', 'MORA'),
(104, 3, 15000000.00, 0.00, 15.00, '2022-11-20', 'PAGADO'),
(105, 4, 30000000.00, 32500000.00, 22.00, '2023-03-12', 'MORA'),
(106, 6, 80000000.00, 65000000.00, 17.00, '2023-05-18', 'VIGENTE');
""")

%sql conn
print('✅ Entorno DuckDB inicializado para el Módulo 02 (Filtrado & Patrones LIKE).')

✅ Entorno DuckDB inicializado para el Módulo 02 (Filtrado & Patrones LIKE).


## 1. Fundamentos Teóricos de la Búsqueda por Patrones
En SQL relacional, el filtrado de texto mediante los operadores `=` o `<>` exige una coincidencia exacta carácter por carácter. Para realizar **búsquedas aproximadas, difusas o basadas en patrones**, se emplean los operadores `LIKE` e `ILIKE`.
### 📐 Tabla Comparativa de Operadores Comodín
| Operador / Comodín | Significado Técnico | Ejemplo de Patrón | Coincide con | No coincide con |
| :--- | :--- | :--- | :--- | :--- |
| `%` (Porcentaje) | Cero, uno o $N$ caracteres cualesquiera | `'Tech%'` | `'Tech'`, `'Technology'`, `'TechCorp'` | `'InverTech'` |
| `%` (Centrado) | Contiene la subcadena en cualquier posición | `'%Valle%'` | `'Construcciones del Valle Ltda.'` | `'Construcciones del Norte'` |
| `_` (Guion bajo) | Exactamente **1 carácter** en esa posición | `'7945612_'` | `'79456123'`, `'79456129'` | `'794561234'` |
| `LIKE` | Comparación sensible a mayúsculas (`Case-Sensitive`) | `'Bogotá'` | `'Bogotá'` | `'bogotá'`, `'BOGOTÁ'` |
| `ILIKE` | Comparación insensible a mayúsculas (`Case-Insensitive`) | `'bogotá'` | `'Bogotá'`, `'BOGOTÁ'`, `'bogotá'` | N/A |
| `ESCAPE` | Permite buscar caracteres comodín literales | `'%50\%% ESCAPE \'` | `'Descuento del 50%'` | `'Descuento del 500'` |

## 2. Ejemplos Prácticos de Patrones LIKE / ILIKE
### 2.1 Búsqueda por Prefijo (`'Texto%'`)
Filtra los clientes cuyo nombre empiece con las palabras `'Inversiones'` o `'Construcciones'`.

In [3]:
%%sql
SELECT *
FROM clientes_cartera

,cliente_id,nombre,doc_identidad,score_credito,cupo_aprobado,ciudad,email
0,1,Inversiones Empresariales Bogotá S.A.S.,900123456-1,820,150000000.0,Bogotá,contacto@inversionesbogota.co
1,2,Construcciones del Valle Ltda.,890987654-3,680,80000000.0,Cali,gerencia@convalle.com
2,3,María Fernanda Gómez,1018456789,750,25000000.0,Medellín,mfgomez@email.com
3,4,Comercializadora Caribe S.A.,800555444-2,590,45000000.0,Barranquilla,ventas@caribesa.co
4,5,Jorge Enrique Restrepo,79456123,420,10000000.0,Bogotá,jrestrepo@email.com
5,6,Tecnología & Soluciones 50% S.A.S.,901999888-7,810,95000000.0,Bogotá,info@tech_sol.com.co
6,7,Servicios Logísticos del Eje Ltda.,830111222-9,710,60000000.0,Pereira,None


In [5]:
%%sql
-- 1. Clientes cuyo nombre comienza exactamente por ''
SELECT cliente_id, nombre, doc_identidad, ciudad
FROM clientes_cartera
WHERE nombre ILIKE 'C%';

,cliente_id,nombre,doc_identidad,ciudad
0,2,Construcciones del Valle Ltda.,890987654-3,Cali
1,4,Comercializadora Caribe S.A.,800555444-2,Barranquilla


### 2.2 Búsqueda por Sufijo (`'%Texto'`)
Identificar sociedades comerciales cuyo tipo societario al final sea `'S.A.S.'` o `'Ltda.'` usando la insensibilidad de `ILIKE`.

In [6]:
%%sql
-- 2. Buscar empresas que terminen en 'S.A.S.' (insensible a mayúsculas/minúsculas con ILIKE)
SELECT cliente_id, nombre, email
FROM clientes_cartera
WHERE nombre ILIKE '%s.a.s.';

,cliente_id,nombre,email
0,1,Inversiones Empresariales Bogotá S.A.S.,contacto@inversionesbogota.co
1,6,Tecnología & Soluciones 50% S.A.S.,info@tech_sol.com.co


### 2.3 Búsqueda con Comodín de Posición Única (`_`)
El carácter `_` valida la longitud y el formato de un campo. Por ejemplo, documentos de identidad que tengan una estructura fija de 9 dígitos seguidos de guion y digito de verificación `900______-_`.

In [8]:
%%sql
-- 3. Filtrar NITs de empresas con formato exacto: 9 dígitos + guion + 1 dígito de verificación
SELECT cliente_id, nombre, doc_identidad
FROM clientes_cartera
WHERE doc_identidad LIKE '_________- _' ESCAPE ' ' OR doc_identidad LIKE '900______-_';

,cliente_id,nombre,doc_identidad
0,1,Inversiones Empresariales Bogotá S.A.S.,900123456-1


### 2.4 Escapar Caracteres Comodín Literales (`ESCAPE`)
¿Cómo buscamos registros que contengan el carácter `%` o `_` como texto literal? Usamos la cláusula `ESCAPE '\'`.

In [9]:
%%sql
-- 4. Buscar la empresa que contiene el carácter literal '%' en su razón social (ej. '50%')
SELECT cliente_id, nombre, email
FROM clientes_cartera
WHERE nombre LIKE '%\%%' ESCAPE '\';

,cliente_id,nombre,email
0,6,Tecnología & Soluciones 50% S.A.S.,info@tech_sol.com.co


## 3. Subconsultas Combinadas con LIKE / ILIKE
Las subconsultas permiten utilizar los resultados de una búsqueda por patrones en una tabla para filtrar o consultar registros en otra tabla vinculada.

### 3.1 Subconsulta en WHERE con Operador `IN` y `LIKE`
**Pregunta de Negocio:** Obtener todos los créditos desembolsados a clientes cuyo correo electrónico pertenezca al dominio colombiano `%.co`.

In [10]:
%%sql
-- Subconsulta: Seleccionar IDs de clientes con email terminado en '.co'
-- Consulta Externa: Obtener los créditos asociados a dichos clientes
SELECT 
    credito_id,
    cliente_id,
    monto_desembolsado,
    saldo_actual,
    estado
FROM creditos
WHERE cliente_id IN (
    SELECT cliente_id
    FROM clientes_cartera
    WHERE email ILIKE '%.co'
);

,credito_id,cliente_id,monto_desembolsado,saldo_actual,estado
0,101,1,100000000.0,75000000.0,VIGENTE
1,102,1,40000000.0,38000000.0,VIGENTE
2,105,4,30000000.0,32500000.0,MORA
3,106,6,80000000.0,65000000.0,VIGENTE


### 3.2 Subconsulta Correlacionada con `EXISTS` y `LIKE`
**Pregunta de Negocio:** Listar los clientes de la cartera que tengan al menos un crédito en estado `'MORA'` y cuya razón social contenga el término `'Valle'` o `'Caribe'`.

In [ ]:
%%sql
-- Subconsulta correlacionada con EXISTS y filtrado LIKE combinando OR
SELECT 
    c.cliente_id,
    c.nombre,
    c.ciudad,
    c.cupo_aprobado
FROM clientes_cartera c
WHERE (c.nombre ILIKE '%valle%' OR c.nombre ILIKE '%caribe%')
  AND EXISTS (
      SELECT cr.credito_id
      FROM creditos cr
      WHERE cr.cliente_id = c.cliente_id
        AND cr.estado = 'MORA'
  );

### 3.3 Subconsulta de Agregación con Filtro `LIKE` en `HAVING`
**Pregunta de Negocio:** Calcular el saldo total prestado agrupado por ciudad, incluyendo solo aquellas ciudades que comiencen con la letra `'B'` (ej. Bogotá, Barranquilla, Bucaramanga).

In [ ]:
%%sql
SELECT 
    c.ciudad,
    COUNT(cr.credito_id) AS total_creditos,
    SUM(cr.monto_desembolsado) AS total_desembolsado,
    SUM(cr.saldo_actual) AS saldo_pendiente
FROM clientes_cartera c
JOIN creditos cr ON c.cliente_id = cr.cliente_id
WHERE c.ciudad ILIKE 'b%'
GROUP BY c.ciudad
HAVING SUM(cr.saldo_actual) > 0;

## 4. Lógica Trivalente & Manejo de Nulos (NULL)
En SQL, una comparación contra un valor `NULL` mediante `LIKE` no resulta en `TRUE` ni en `FALSE`, sino en **`UNKNOWN`**.
Como las cláusulas `WHERE` descartan cualquier fila que evalúe a `UNKNOWN`, los registros con `NULL` son excluidos silenciosamente. Para resolverlo se utiliza **`COALESCE`**.

In [ ]:
%%sql
-- Demostración de COALESCE: Garantiza que si email es NULL, se reemplace por '' para evitar evaluación UNKNOWN
SELECT 
    cliente_id,
    nombre,
    email,
    COALESCE(email, 'SIN_REGISTRO') AS email_seguro
FROM clientes_cartera
WHERE COALESCE(email, '') NOT ILIKE '%.com';

## 5. Retos Prácticos de Filtrado & LIKE para el Estudiante 🏋️‍♂️
### ✏️ Reto 1: Filtrado con Comodines Múltiples
Escribe una consulta SQL que devuelva los clientes cuyo NIT empiece por `900` o `890` y cuyo nombre contenga la palabra `'Ltda.'` o `'S.A.S.'`.
### ✏️ Reto 2: Subconsulta con LIKE e Indicador Financiero
Encuentra el monto desembolsado promedio de los créditos concedidos a clientes cuyo correo electrónico pertenezca a dominios comerciales `.com`.
### ✏️ Reto 3: Comodín de Posición Exacta (`_`)
Selecciona los clientes cuyo número de documento tenga exactamente 8 caracteres antes del guion de verificación.

In [ ]:
%%sql
-- 🛠 Espacio para la solución del estudiante:

-- Reto 1: SELECT ... FROM clientes_cartera WHERE ...

-- Reto 2: SELECT AVG(monto_desembolsado) FROM creditos WHERE cliente_id IN (...)

-- Reto 3: SELECT ... FROM clientes_cartera WHERE doc_identidad LIKE ...;

In [ ]:
%%sql
-- ==============================================================================
-- 🛠 SOLUCIÓN A LOS RETOS PRÁCTICOS DEL MÓDULO 02 (LIKE & SUBCONSULTAS)
-- ==============================================================================

-- ------------------------------------------------------------------------------
-- ✏️ RETO 1: Filtrado con Comodines Múltiples (% y Operadores Lógicos)
-- NITs que inicien por '900' o '890' y cuyo nombre termine o contenga 'Ltda.' o 'S.A.S.'
-- ------------------------------------------------------------------------------
SELECT 
    cliente_id, 
    nombre, 
    doc_identidad, 
    ciudad, 
    email
FROM clientes_cartera
WHERE (doc_identidad LIKE '900%' OR doc_identidad LIKE '890%')
  AND (nombre ILIKE '%Ltda.%' OR nombre ILIKE '%S.A.S.%');


-- ------------------------------------------------------------------------------
-- ✏️ RETO 2: Subconsulta con LIKE e Indicador Financiero
-- Monto desembolsado promedio de los créditos a clientes con correo en dominios '.com'
-- ------------------------------------------------------------------------------
SELECT 
    ROUND(AVG(monto_desembolsado), 2) AS promedio_desembolsado,
    COUNT(credito_id) AS total_creditos_evaluados
FROM creditos
WHERE cliente_id IN (
    SELECT cliente_id
    FROM clientes_cartera
    WHERE email ILIKE '%.com%'
);


-- ------------------------------------------------------------------------------
-- ✏️ RETO 3: Comodín de Posición Exacta (_)
-- Buscar documentos de identidad con formato exacto de 9 dígitos + guion + dígito verificador
-- ------------------------------------------------------------------------------
SELECT 
    cliente_id, 
    nombre, 
    doc_identidad
FROM clientes_cartera
WHERE doc_identidad LIKE '_________-_';
